# PDBBind subset preparation (benchmark input)

Moved out of the single-example notebook. Single-complex docking (`02_docking_colab.ipynb`) needs **none** of this.

This notebook prepares only the PDBBind complexes **you list** (or the first *N* of an upstream split):
the 2 GB archive is downloaded once, but only the requested complexes are unpacked, and the protein FASTA and
ESM2 embeddings are computed for those complexes alone (the old notebook embedded all 24 926 sequences).

Outputs go to `<workdir>/benchmark/pdbbind_subset_<N>_<hash>/` (the hash identifies the exact set of ids):
`PDBBind_processed/<id>/`, `prepared_for_esm.fasta`, `esm2_embeddings.pt`, `split_subset.txt`, `subset_manifest.json`.

In [ ]:
#@title 1. Get the helper code
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/nassim-arifette/diffdock-geometric-diffusion-docking.git"  #@param {type:"string"}
REPO_REF = "main"  #@param {type:"string"}

IN_COLAB = "google.colab" in sys.modules
here = Path.cwd()
checkout = next((p for p in (here, here.parent) if (p / "docking_tools").is_dir()), None)
if checkout is None:  # fresh Colab VM: fetch the repository that contains docking_tools/
    checkout = Path("/content/diffdock-geometric-diffusion-docking")
    if not checkout.exists():
        subprocess.run(["git", "clone", "--quiet", "--branch", REPO_REF, REPO_URL, str(checkout)], check=True)
    else:  # a checkout from an earlier cell run: pick up newer commits instead of silently using stale code
        for args in (["fetch", "--quiet", "origin", REPO_REF], ["checkout", "--quiet", REPO_REF], ["merge", "--ff-only", "--quiet", "FETCH_HEAD"]):
            if subprocess.run(["git", "-C", str(checkout), *args]).returncode != 0:
                print("warning: could not update", checkout, "- using it as it is")
                break
if str(checkout) not in sys.path:
    sys.path.insert(0, str(checkout))

if IN_COLAB:  # the notebook's own Python only needs these; DiffDock runs in its own pinned environment
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", ['rdkit==2026.9.1', 'py3Dmol==2.5.5', 'pyyaml']], check=True)

import docking_tools
print("docking_tools", docking_tools.__version__, "from", checkout)

In [ ]:
#@title 2. Which complexes?
IDS = "1a0q"  #@param {type:"string"}
#@markdown Space- or comma-separated PDB ids. Leave empty to take the first N ids of an upstream split file instead.
SPLIT_FILE = "data/splits/timesplit_test"  #@param {type:"string"}
LIMIT = 20  #@param {type:"integer"}
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/DiffDock_colab"  #@param {type:"string"}
WORKDIR = os.environ.get("DIFFDOCK_WORKDIR", "/content/diffdock_work")
DRIVE_DIR = os.environ.get("DOCKING_DRIVE_ROOT", DRIVE_DIR)

from docking_tools.drive import ensure_drive
drive_root = ensure_drive(DRIVE_DIR) if SAVE_TO_DRIVE else None

In [ ]:
#@title 3. Environment (same pinned setup as the docking notebook)
from docking_tools.setup_env import setup

setup(WORKDIR, cache_dir=drive_root / "cache" if drive_root else None)

In [ ]:
#@title 4. Prepare the subset
from docking_tools import benchmark_prep as bp
from docking_tools.setup_env import Layout

layout = Layout(Path(WORKDIR).resolve())
ids = [i for i in IDS.replace(",", " ").split()] or bp.read_split_ids(layout.repo / SPLIT_FILE, LIMIT)
print(len(ids), "complexes:", ids[:10], "..." if len(ids) > 10 else "")
summary = bp.prepare_subset(WORKDIR, ids)
print({k: summary[k] for k in ("n_complexes", "n_sequences", "data_dir", "embeddings", "split_file")})

if drive_root:  # the dataset (complexes + FASTA + embeddings) is saved automatically
    from docking_tools.drive import persist_dataset
    print("saved to:", persist_dataset(Path(summary["data_dir"]).parent, drive_root, Path(summary["data_dir"]).parent.name))